# Neo4j + OpenAI Integration for Math Problem Solver

This notebook implements the core functionality for a math problem solver that:
1. Connects to a Neo4j database to access curriculum skills
2. Uses OpenAI API to generate solutions
3. Matches solution steps to curriculum skills using embeddings

## 1. Install Dependencies

In [1]:
%pip install neo4j openai python-dotenv pandas numpy scikit-learn


  Using cached neo4j-5.28.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached python_dotenv-1.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached pandas-2.2.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (89 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 997.5 kB/s eta 0:00:00 0:00:01
  Using cached scikit_learn-1.6.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (31 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 2.2 MB/s eta 0:00:00a 0:00:01
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.13.2-py3-none-any.whl.metadata (3.0 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.15.2-cp312-cp312-macosx_14_0_arm64.whl.metad

## 2. Environment Setup

In [3]:

import os
import json
import numpy as np
from typing import List, Dict, Any
from dotenv import load_dotenv
from neo4j import GraphDatabase
import openai
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load environment variables
load_dotenv()

# Neo4j configuration
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

# OpenAI configuration
openai.api_key = os.getenv("OPENAI_API_KEY")

print(f"Neo4j URI: {NEO4J_URI}")
print(f"OpenAI API Key configured: {'Yes' if openai.api_key else 'No'}")







Neo4j URI: bolt://localhost:7687
OpenAI API Key configured: Yes


## 3. Neo4j Database Connection

In [4]:

# In[3]:
class Neo4jConnection:
    def __init__(self, uri: str, user: str, password: str):
        """Initialize connection to Neo4j database"""
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        
    def close(self):
        """Close the connection"""
        self.driver.close()
        
    def query(self, query: str, parameters=None):
        """Execute a Cypher query and return results as records"""
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [record for record in result]
        
    def query_to_dataframe(self, query: str, parameters=None):
        """Execute a Cypher query and return results as a pandas DataFrame"""
        with self.driver.session() as session:
            result = session.run(query, parameters)
            data = [dict(record) for record in result]
            return pd.DataFrame(data) if data else pd.DataFrame()

# Create connection instance
try:
    conn = Neo4jConnection(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    print("Connected to Neo4j successfully")
except Exception as e:
    print(f"Failed to connect to Neo4j: {e}")


Connected to Neo4j successfully


## 4. Database Schema and Skills Retrieval


In [5]:

# In[4]:
def get_curriculum_structure():
    """Get the overall structure of the curriculum from Neo4j"""
    chapters_query = """
    MATCH (c:Chapter)
    RETURN c.elementID as id, c.number as number, c.name as name, c.level as level
    ORDER BY c.level, c.number
    """
    
    requirements_query = """
    MATCH (c:Chapter)-[:HAS_REQUIREMENT]->(r:Requirement)
    RETURN c.elementID as chapter_id, r.elementID as id, r.number as number, 
           r.name as name, r.level as level
    ORDER BY c.level, c.number, r.number
    """
    
    skills_query = """
    MATCH (r:Requirement)-[:HAS_GOAL]->(s:Goal)
    RETURN r.elementID as req_id, s.elementID as id, 
           s.text as text,
    """
    
    chapters = conn.query_to_dataframe(chapters_query)
    requirements = conn.query_to_dataframe(requirements_query)
    skills = conn.query_to_dataframe(skills_query)
    
    return {
        "chapters": chapters,
        "requirements": requirements,
        "skills": skills
    }

def get_skills_by_grade(grade: str):
    """Get skills filtered by grade"""
    query = """
    MATCH (s:Skill)
    WHERE s.grade = $grade
    RETURN s.elementID as id, s.name as name, s.text as text, s.type as type
    """
    return conn.query_to_dataframe(query, parameters={"grade": grade})


## 5. OpenAI API Integration

In [6]:
# In[5]:
class OpenAIService:
    def __init__(self, api_key: str):
        """Initialize OpenAI service"""
        openai.api_key = api_key
        self.embedding_model = "text-embedding-ada-002"
        self.chat_model = "gpt-4-turbo"
    
    def generate_solution(self, problem: str) -> Dict:
        """Generate a step-by-step solution for a math problem"""
        try:
            response = openai.ChatCompletion.create(
                model=self.chat_model,
                messages=[
                    {"role": "system", "content": """You are a helpful math tutor. 
                     Generate a step-by-step solution for the math problem. 
                     For each step, provide:
                     1. A hint that helps without giving away the full solution
                     2. The complete solution for the step
                     3. A list of skills needed to solve this step
                     
                     Format your response as a valid JSON with the following structure:
                     {
                         "steps": [
                             {
                                 "step_number": 1,
                                 "hint": "A hint that helps without giving away the full solution",
                                 "solution": "The complete solution for this step",
                                 "skills": ["skill1", "skill2"]
                             },
                             ...
                         ]
                     }
                     """},
                    {"role": "user", "content": f"Problem: {problem}"}
                ],
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            print(f"Error generating solution: {e}")
            return {"error": str(e)}

    def get_embedding(self, text: str) -> List[float]:
        """Get embedding vector for text using OpenAI's embedding model"""
        response = openai.Embedding.create(
            model=self.embedding_model,
            input=text
        )
        return response["data"][0]["embedding"]
    
    def verify_skills(self, solution_with_matches: Dict) -> Dict:
        """Verify matched skills for each solution step"""
        try:
            response = openai.ChatCompletion.create(
                model=self.chat_model,
                messages=[
                    {"role": "system", "content": """You are a curriculum expert.
                     Review the math solution steps and the matched skills from the curriculum.
                     For each step, select only the skills that are truly relevant.
                     Return your response in the same JSON format, but include only the verified skills."""},
                    {"role": "user", "content": f"Solution with matched skills: {json.dumps(solution_with_matches)}"}
                ],
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            print(f"Error verifying skills: {e}")
            return {"error": str(e)}


## 6. Skill Matcher using Embeddings

In [9]:

class SkillMatcher:
    def __init__(self, openai_service: OpenAIService):
        """Initialize the skill matcher with OpenAI service"""
        self.openai_service = openai_service
        self.skill_embeddings = {}
        
    def create_skill_embeddings(self, skills_df: pd.DataFrame):
        """Create embeddings for all skills in the dataframe"""
        print("Creating skill embeddings...")
        for _, skill in skills_df.iterrows():
            skill_text = f"{skill['name']}. {skill['text']}"
            self.skill_embeddings[skill['id']] = {
                'id': skill['id'],
                'name': skill['name'],
                'text': skill['text'],
                'embedding': self.openai_service.get_embedding(skill_text)
            }
        print(f"Created embeddings for {len(self.skill_embeddings)} skills")
    
    def match_skills_for_step(self, step_text: str, skills_list: List[str], top_k: int = 3):
        """Match the most relevant skills from the database for a solution step"""
        step_embedding = self.openai_service.get_embedding(step_text)
        
        similarities = []
        for skill_id, skill_data in self.skill_embeddings.items():
            similarity = cosine_similarity(
                [step_embedding], 
                [skill_data['embedding']]
            )[0][0]
            similarities.append({
                'id': skill_id,
                'name': skill_data['name'],
                'text': skill_data['text'],
                'similarity': similarity
            })
        
        # Sort by similarity and get top_k
        top_matches = sorted(similarities, key=lambda x: x['similarity'], reverse=True)[:top_k]
        return top_matches
    
    def match_skills_for_solution(self, solution: Dict, top_k: int = 3):
        """Match skills for all steps in a solution"""
        solution_with_matches = {"steps": []}
        
        for step in solution["steps"]:
            step_text = f"{step['hint']} {step['solution']}"
            matched_skills = self.match_skills_for_step(step_text, step['skills'], top_k)
            
            solution_with_matches["steps"].append({
                "step_number": step["step_number"],
                "hint": step["hint"],
                "solution": step["solution"],
                "skills": {
                    "skill_desc": step["skills"],
                    "retrieved_skills": [
                        {"neo4jid": match["id"], "skill_desc": f"{match['name']}: {match['text']}"}
                        for match in matched_skills
                    ]
                }
            })
            
        return solution_with_matches


## 7. Solver Service

In [10]:


# In[7]:
class SolverService:
    def __init__(self, openai_service: OpenAIService, skill_matcher: SkillMatcher):
        """Initialize the solver service"""
        self.openai_service = openai_service
        self.skill_matcher = skill_matcher
    
    def solve_problem(self, problem: str, grade: str = "4-6") -> Dict:
        """Solve a math problem with steps and matched skills"""
        # Step 1: Generate solution with OpenAI
        solution = self.openai_service.generate_solution(problem)
        
        if "error" in solution:
            return solution
        
        # Step 2: Match skills to solution steps
        solution_with_matches = self.skill_matcher.match_skills_for_solution(solution)
        
        # Step 3: Verify skills with OpenAI
        verified_solution = self.openai_service.verify_skills(solution_with_matches)
        
        return verified_solution


## 8. Test the Implementation

In [11]:


# In[8]:
def main():
    """Main function to test the implementation"""
    # Initialize services
    openai_service = OpenAIService(openai.api_key)
    skill_matcher = SkillMatcher(openai_service)
    
    # Load skills from Neo4j
    try:
        # Get curriculum structure
        curriculum = get_curriculum_structure()
        print(f"Retrieved {len(curriculum['skills'])} skills from Neo4j")
        
        # Create embeddings for skills (only for demonstration - in production, this would be done once and cached)
        if not curriculum['skills'].empty:
            skill_matcher.create_skill_embeddings(curriculum['skills'])
            
            # Initialize solver service
            solver_service = SolverService(openai_service, skill_matcher)
            
            # Test with a sample problem
            sample_problem = "If a rectangle has a length of 8 cm and a width of 5 cm, what is its area?"
            print(f"\nSolving problem: {sample_problem}")
            
            solution = solver_service.solve_problem(sample_problem)
            print("\nSolution with matched skills:")
            print(json.dumps(solution, indent=2))
        else:
            print("No skills found in the database. Please check your Neo4j setup.")
    except Exception as e:
        print(f"Error in main execution: {e}")
    finally:
        # Close Neo4j connection
        conn.close()

# Run the main function if this is the entry point
if __name__ == "__main__":
    main()

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: elementID)} {position: line: 3, column: 14, offset: 36} for query: '\n    MATCH (c:Chapter)\n    RETURN c.elementID as id, c.number as number, c.name as name, c.level as level\n    ORDER BY c.level, c.number\n    '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the l

Error in main execution: {code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input '': expected an expression (line 5, column 5 (offset: 134))
""
     ^}
